# Enterprise Data Architecture & Reconciliation

## CFO Financial Audit Dashboard

This project reconciles three separate data sources: a historical customer database, raw server logs, and daily EUR-to-USD exchange rates.

### Approach
- Use a SQL window function to keep the latest customer status.
- Remove `ERROR` log rows and extract transaction fields with vectorized Regex.
- Forward-fill exchange rates so weekend transactions retain the previous available rate.
- Keep transactions belonging to customers whose latest status is Active.
- Calculate USD revenue and summarize it by month and customer.

The main processing is vectorized; no Pandas row-by-row `for` loop is used.


In [ ]:
# ===================================================================
# PHASE 1: SQL EXTRACTION
# ===================================================================
import sqlite3
import pandas as pd
import re
import matplotlib.pyplot as plt

db_path = "enterprise_database.db"

con = sqlite3.connect(db_path)

sql = """
WITH ranked_users AS (
    SELECT user_id, name, status, updated_at,
           ROW_NUMBER() OVER (
               PARTITION BY user_id
               ORDER BY updated_at DESC
           ) AS rn
    FROM users
)
SELECT user_id, name, status, updated_at
FROM ranked_users
WHERE rn = 1
  AND status = 'Active'
"""

df_active_users = pd.read_sql_query(sql, con)
con.close()

print("Active users:", len(df_active_users))
df_active_users.head()


### Why this SQL works
`ROW_NUMBER()` orders each user's historical records from newest to oldest. `rn = 1` therefore keeps only the latest record, and the final filter keeps users whose latest status is Active.


In [ ]:
# ===================================================================
# PHASE 2: REGEX & UNSTRUCTURED DATA WRANGLING
# ===================================================================
log_path = "server_logs.txt"

logs = pd.read_csv(log_path, header=0, names=["Raw_Server_Log"])
total_log_rows = len(logs)

# ERROR records represent failed transactions and are excluded.
logs = logs.loc[~logs["Raw_Server_Log"].str.contains("ERROR", na=False)].copy()

pattern = (
    r'\[(\d{4}-\d{2}-\d{2})\s+\d{2}:\d{2}:\d{2}\]'
    r'.*?payload=\{"u":"([^"]+)",\s*"item_code":"([^"]+)",'
    r'\s*"eur_val":([0-9]+(?:\.[0-9]+)?)\}'
)

extracted_data = logs["Raw_Server_Log"].str.extract(pattern)
extracted_data.columns = ["Date", "User_ID", "Product_ID", "EUR_Value"]
extracted_data["Date"] = pd.to_datetime(extracted_data["Date"])
extracted_data["EUR_Value"] = pd.to_numeric(extracted_data["EUR_Value"])

print("Original log rows:", total_log_rows)
print("Rows after removing ERROR logs:", len(extracted_data))
print("Regex extraction complete:", extracted_data.notna().all().all())
extracted_data.head()


In [ ]:
# ===================================================================
# PHASE 3: FINANCIAL RECONCILIATION & MERGING
# ===================================================================
rates = pd.read_csv("daily_exchange_rates.csv")
rates["Date"] = pd.to_datetime(rates["Date"])
rates = rates.sort_values("Date")

# The rate table has missing weekend values. Carry Friday's rate forward.
rates["EUR_to_USD"] = rates["EUR_to_USD"].ffill()

df_final = extracted_data.merge(rates, on="Date", how="left")
df_final = df_final.merge(
    df_active_users[["user_id", "name"]],
    left_on="User_ID", right_on="user_id", how="inner"
).drop(columns=["user_id"])

df_final["USD_Revenue"] = df_final["EUR_Value"] * df_final["EUR_to_USD"]

print("Final reconciled rows:", len(df_final))
print("Missing exchange rates:", df_final["EUR_to_USD"].isna().sum())
print("Total USD revenue: ${:,.2f}".format(df_final["USD_Revenue"].sum()))
df_final.head()


### Reconciliation note
The final table is an inner join to the Active customer list, so transactions belonging to customers whose latest database status is not Active are not included in the CFO revenue view. This is intentional and follows the project instructions.


In [ ]:
# ===================================================================
# PHASE 4: AGGREGATION & VISUALIZATION
# ===================================================================
monthly_revenue = (
    df_final.assign(Month=df_final["Date"].dt.to_period("M").astype(str))
    .groupby("Month", as_index=False)["USD_Revenue"].sum()
)

top_5_customers = (
    df_final.groupby(["User_ID", "name"], as_index=False)["USD_Revenue"].sum()
    .sort_values("USD_Revenue", ascending=False)
    .head(5)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(monthly_revenue["Month"], monthly_revenue["USD_Revenue"], marker="o")
axes[0].set_title("Total USD Revenue by Month")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("USD Revenue")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(alpha=0.25)

labels = top_5_customers["User_ID"] + "\n" + top_5_customers["name"]
axes[1].bar(labels, top_5_customers["USD_Revenue"])
axes[1].set_title("Top 5 Most Valuable Customers")
axes[1].set_xlabel("Customer")
axes[1].set_ylabel("USD Revenue")
axes[1].tick_params(axis="x", rotation=30)
axes[1].grid(axis="y", alpha=0.25)

fig.suptitle("CFO Financial Audit Dashboard", fontsize=14)
fig.tight_layout()
plt.show()

print("Top 5 customers")
display(top_5_customers)


## Final project checks
- 100,000 raw log rows were supplied.
- ERROR log records were removed before Regex extraction.
- All four required transaction fields were extracted successfully.
- Exchange rates were forward-filled before the date join.
- Latest customer status was selected with a SQL window function.
- USD revenue was calculated with vectorized arithmetic.
- The final dashboard contains the two required views.

### Files generated
- `active_users.csv`
- `extracted_transactions.csv`
- `final_reconciled_data.csv`
- `monthly_revenue.csv`
- `top_5_customers.csv`
- `cfo_dashboard.png`
